# 03 — KnottedGraph vs Topoly: invariant and full-pipeline scaling

This notebook reports **two distinct benchmarks** for the same spatial-graph families:

1. **Identical-PD Yamada timing**: the spatial graph is converted to a PD code once outside the timer. KnottedGraph and Topoly then receive the **same PD string**, and only Yamada evaluation is timed.
2. **Spatial graph → PD → Yamada timing**: each timed repetition starts from the same spatial graph, rebuilds its PD representation, and then evaluates Yamada. Both sides deliberately use the same KnottedGraph `PDCode` preprocessing, so this measures **common graph→PD preprocessing + the respective Yamada engine**. It is not presented as Topoly's native spatial-graph pipeline.

Every paired timing is accepted only if both frameworks use the same PD string and their Laurent polynomials agree up to
\[
P_{\rm Topoly}(A)=\pm A^k P_{\rm KG}(A^{\pm1}).
\]

The benchmark spans crossing count, edge count, vertex/input size, and harder connected trivalent graphs. Timeout points are shown as censored observations, not fabricated runtimes. Fitted power-law/exponential curves are empirical scaling summaries, not mathematical Big-\(O\) proofs.


In [ ]:
from pathlib import Path
import csv, json, os, subprocess, sys
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "knotted_graph").exists():
    raise RuntimeError("Run this notebook from inside the KnottedGraph checkout.")

SRC = ROOT / "src"
sys.path.insert(0, str(SRC))

import knotted_graph
kg_path = Path(knotted_graph.__file__).resolve()
if SRC not in kg_path.parents:
    raise RuntimeError(f"A stale knotted_graph was imported from {kg_path}")

try:
    import topoly
except ImportError as exc:
    raise ImportError("Install Topoly first: pip install topoly") from exc

OUT = ROOT / "User_guide" / "benchmarks"
RES = OUT / "results_latest"
FIG = OUT / "figures_latest"
RES.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)

print("KnottedGraph:", kg_path)
print("Topoly:", Path(topoly.__file__).resolve())


## 1. Run both correctness-gated benchmark modes


In [ ]:
script = ROOT / "dev" / "benchmark_topoly_extended_scaling.py"
env = dict(os.environ)
env["PYTHONPATH"] = str(SRC)
env["PYTHONNOUSERSITE"] = "1"

proc = subprocess.run(
    [sys.executable, str(script), "--timeout", "10"],
    cwd=ROOT,
    env=env,
    text=True,
    capture_output=True,
    timeout=3600,
)
print(proc.stdout)

if proc.returncode:
    raise RuntimeError(
        f"Extended Topoly benchmark failed with exit code {proc.returncode}.\n"
        f"STDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}"
    )

for line in reversed(proc.stdout.splitlines()):
    if line.startswith("SUMMARY="):
        rows = json.loads(line[8:])
        break
else:
    raise RuntimeError("Benchmark completed without SUMMARY output.")

keys = list(dict.fromkeys(key for row in rows for key in row))
with (RES / "topoly_extended_scaling.csv").open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=keys)
    writer.writeheader()
    writer.writerows(rows)

families = {}
for row in rows:
    families.setdefault(row["family"], []).append(row)

for family, fr in families.items():
    inv = sum(
        r["invariant_knottedgraph_status"] == "ok"
        and r["invariant_topoly_status"] == "ok"
        for r in fr
    )
    pipe = sum(
        r["pipeline_knottedgraph_status"] == "ok"
        and r["pipeline_topoly_status"] == "ok"
        for r in fr
    )
    print(
        f"{family:22s}: identical-PD {inv}/{len(fr)} | "
        f"full-pipeline {pipe}/{len(fr)}"
    )


## 2. Plotting and empirical-fit helpers


In [ ]:
def series(rows_, xkey, framework, mode):
    xs, ys, censored = [], [], []
    for row in sorted(rows_, key=lambda r: r[xkey]):
        xs.append(float(row[xkey]))
        status = row[f"{mode}_{framework}_status"]
        if status == "ok":
            ys.append(float(row[f"{mode}_{framework}_s"]))
            censored.append(False)
        elif status == "timeout":
            ys.append(float(row["timeout_s"]))
            censored.append(True)
        else:
            ys.append(np.nan)
            censored.append(True)
    return np.asarray(xs), np.asarray(ys), np.asarray(censored)


def fits(xs, ys, censored):
    mask = (~censored) & np.isfinite(ys) & (ys > 0) & (xs > 0)
    x, y = xs[mask], ys[mask]
    if len(x) < 3:
        return {}
    ly, lx = np.log(y), np.log(x)
    alpha, log_cp = np.polyfit(lx, ly, 1)
    beta, log_ce = np.polyfit(x, ly, 1)

    def r2(actual, predicted):
        denom = np.sum((actual - actual.mean()) ** 2)
        return 1 - np.sum((actual - predicted) ** 2) / denom if denom else 1.0

    return {
        "alpha": float(alpha),
        "Cp": float(np.exp(log_cp)),
        "power_r2": float(r2(ly, log_cp + alpha * lx)),
        "beta": float(beta),
        "Ce": float(np.exp(log_ce)),
        "exp_r2": float(r2(ly, log_ce + beta * x)),
    }


def plot_family(rows_, xkey, xlabel, title, stem, *, mode, logx=False):
    plt.figure(figsize=(9.6, 5.8))
    reports = {}

    for framework, label, marker in [
        ("knottedgraph", "KnottedGraph", "o"),
        ("topoly", "Topoly", "s"),
    ]:
        xs, ys, cens = series(rows_, xkey, framework, mode)
        ok = (~cens) & np.isfinite(ys)
        timeout = cens & np.isfinite(ys)

        plt.plot(xs[ok], ys[ok], marker=marker, label=label)
        if timeout.any():
            plt.scatter(
                xs[timeout],
                ys[timeout],
                marker=marker,
                facecolors="none",
                label=f"{label}: timeout/censored",
            )

        report = fits(xs, ys, cens)
        reports[framework] = report
        if report and ok.any():
            xx = (
                np.geomspace(xs[ok].min(), xs[ok].max(), 200)
                if logx
                else np.linspace(xs[ok].min(), xs[ok].max(), 200)
            )
            yy = report["Cp"] * xx ** report["alpha"]
            plt.plot(
                xx,
                yy,
                linestyle="--",
                label=(
                    f"{label} power fit: alpha={report['alpha']:.2f}, "
                    f"R2={report['power_r2']:.3f}"
                ),
            )

    plt.yscale("log")
    if logx:
        plt.xscale("log")
    plt.xlabel(xlabel)
    plt.ylabel(
        "Yamada evaluation time (s)"
        if mode == "invariant"
        else "Graph → PD → Yamada time (s)"
    )
    plt.title(title)
    plt.grid(alpha=0.25)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(FIG / f"{stem}.png", dpi=300, bbox_inches="tight")
    plt.savefig(FIG / f"{stem}.pdf", bbox_inches="tight")
    plt.show()

    for framework, report in reports.items():
        if report:
            print(
                f"{framework}: power alpha={report['alpha']:.4g}, "
                f"R2={report['power_r2']:.4f}; "
                f"exp beta={report['beta']:.4g}, R2={report['exp_r2']:.4f}"
            )


def plot_both(rows_, xkey, xlabel, base_title, stem, *, logx=False):
    print("\nIDENTICAL-PD YAMADA ONLY")
    plot_family(
        rows_,
        xkey,
        xlabel,
        base_title + " — identical-PD Yamada",
        stem + "_invariant",
        mode="invariant",
        logx=logx,
    )

    print("\nFULL SPATIAL-GRAPH WORKFLOW")
    plot_family(
        rows_,
        xkey,
        xlabel,
        base_title + " — graph→PD→Yamada",
        stem + "_pipeline",
        mode="pipeline",
        logx=logx,
    )


## 3. Crossing complexity at fixed \(V=2,E=3\)

This isolates crossing-state growth: the abstract graph stays fixed while only the projected crossing count changes.


In [ ]:
plot_both(
    families["crossings_fixed"],
    "crossings",
    "Projected crossings, c",
    "Crossing complexity at fixed V=2, E=3",
    "topoly_vs_knottedgraph_crossings_fixed",
)


## 4. Large-\(c\) diagram throughput

Independent one-crossing theta components target much larger crossing counts. Because \(V\) and \(E\) grow with \(c\), this is a throughput curve rather than a pure crossing-complexity curve.


In [ ]:
plot_both(
    families["crossings_throughput"],
    "crossings",
    "Projected crossings, c",
    "Large-diagram throughput versus crossing count",
    "topoly_vs_knottedgraph_crossings_throughput",
)


## 5. Edge scaling to \(E=200\)

The parallel-edge theta family keeps \(V=2,c=0\), isolating edge-count growth.


In [ ]:
plot_both(
    families["edges_theta"],
    "E",
    "Graph edges, E",
    "Edge scaling at fixed V=2, c=0",
    "topoly_vs_knottedgraph_edges",
    logx=True,
)


## 6. Trivalent vertex/input-size scaling to \(V=512\)

Disjoint planar \(K_4\) components remain in the trivalent regime supported by both frameworks. Here \(c=0\) and \(E=3V/2\), so this is sparse input-size scaling rather than an isolated \(V\)-only experiment.


In [ ]:
plot_both(
    families["vertices_k4"],
    "V",
    "Graph vertices, V",
    "Trivalent input-size scaling using planar K4 components",
    "topoly_vs_knottedgraph_vertices_k4",
    logx=True,
)


## 7. Hard connected trivalent scaling

Prism graphs are connected and trivalent with \(V=2n,E=3n\). These cases exercise a harder connected recurrence.


In [ ]:
prism = families["connected_prism"]

plot_both(
    prism,
    "V",
    "Graph vertices, V",
    "Connected trivalent prism scaling with V",
    "topoly_vs_knottedgraph_prism_V",
    logx=True,
)
plot_both(
    prism,
    "E",
    "Graph edges, E",
    "Connected trivalent prism scaling with E",
    "topoly_vs_knottedgraph_prism_E",
    logx=True,
)


## 8. Paired correctness and speed ratios


In [ ]:
for family in [
    "crossings_fixed",
    "crossings_throughput",
    "edges_theta",
    "vertices_k4",
    "connected_prism",
]:
    print(f"\n[{family}]")
    for row in families[family]:
        print(
            f"V={row.get('V','-'):>4} E={row.get('E','-'):>4} "
            f"c={row.get('crossings','-'):>3}"
        )
        for mode, label in [
            ("invariant", "PD→Yamada"),
            ("pipeline", "Graph→PD→Yamada"),
        ]:
            if row[f"{mode}_correctness"] == "PASS":
                print(
                    f"  {label:18s} "
                    f"KG={row[f'{mode}_knottedgraph_s']:.6g}s "
                    f"Topoly={row[f'{mode}_topoly_s']:.6g}s "
                    f"Topoly/KG={row[f'{mode}_topoly_over_kg']:.4g}x PASS"
                )
            else:
                print(
                    f"  {label:18s} "
                    f"KG={row[f'{mode}_knottedgraph_status']} "
                    f"Topoly={row[f'{mode}_topoly_status']}"
                )

for row in rows:
    for mode in ("invariant", "pipeline"):
        if (
            row[f"{mode}_knottedgraph_status"] == "ok"
            and row[f"{mode}_topoly_status"] == "ok"
        ):
            assert row[f"{mode}_correctness"] == "PASS"

print("\nAll completed paired comparisons passed exact polynomial agreement.")


## Interpretation

The **identical-PD** plots isolate the invariant engines. The **graph→PD→Yamada** plots answer the user-workflow question: starting from the same spatial graph, how long does common PD generation plus each Yamada engine take?

The second comparison intentionally uses KnottedGraph's `PDCode` implementation for both sides. This prevents a preprocessing mismatch from contaminating the Yamada comparison and is explicitly **not** a benchmark of Topoly's own spatial-graph wrapper.
